# **Baseline Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
group_name = "Group 24"
student_name = "Mukesh Murugesan"
student_id = "25747763"

In [ ]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [ ]:
import pandas as pd
import altair as alt
import numpy as np
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

---
## A. Assess Baseline Model

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

### A.1 Generate Predictions with Baseline Model

In [ ]:
# ── CONFIRM DATA LOADED CORRECTLY ───────────────────────────────
print("=== LOADED DATA SHAPES ===")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")
print()
print("Features:", X_train.columns.tolist())
print()
print("Target (y_train) — first 5 values (log scale):")
print(y_train.head())
print()
print("Note: y values are in LOG scale (log1p applied in Preparation notebook).")
print("All metrics will be back-transformed to USD using expm1() for reporting.")

In [ ]:
# ── VISUALISE BASELINE: FLAT PREDICTION vs ACTUAL ────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(100), y_val_usd[:100],
        color='steelblue', linewidth=1.2, label='Actual line_total')
ax.axhline(y=np.expm1(y_pred_val_log[0]),
           color='tomato', linestyle='--', linewidth=2,
           label=f'Baseline prediction (${np.expm1(y_pred_val_log[0]):.2f} flat)')
ax.set_title('Baseline Model — Flat Prediction vs Actual (first 100 samples)',
             fontsize=12)
ax.set_xlabel('Sample Index')
ax.set_ylabel('Line Total (USD)')
ax.legend()
plt.tight_layout()
plt.show()
print("The flat red line shows the baseline predicts the same value for every order.")
print("The blue line shows how much line_total actually varies — confirming the")
print("baseline is inadequate and a proper regression model is needed.")

In [ ]:
# ── BUSINESS CONTEXT FOR BASELINE ───────────────────────────────
# The baseline model establishes the performance FLOOR for this project.
# Business Problem: Can we predict line_total better than simply guessing
# the average value for every order?
#
# Baseline hypothesis: A naive model predicting the training mean for all
# inputs will perform poorly (R2=0.00) because line_total varies enormously
# across product categories (Bikes avg $1,744 vs Accessories avg $16).
# Any real ML model must outperform this floor.
#
# Target for Regression Notebook C:
#   R2 > 0.80 | USD RMSE < $700 | USD MAE < $300

print("Baseline establishes performance floor for Notebook C.")
print("Target to beat: R2 > 0.80 | RMSE < $700 | MAE < $300")

In [ ]:
# ── BASELINE MODEL: DummyRegressor (mean strategy) ───────────────
# A baseline model makes the simplest possible prediction:
# predict the MEAN of the training target for every single input.
# This sets the performance floor — any real model must beat this.

# Step 1: Flatten y_train from DataFrame to 1D array
y_train_1d = y_train.values.ravel()
y_val_1d   = y_val.values.ravel()
y_test_1d  = y_test.values.ravel()

# Step 2: Fit baseline — learns only the mean of y_train
baseline_model = DummyRegressor(strategy='mean')
baseline_model.fit(X_train, y_train_1d)

# Step 3: Generate predictions on validation set
y_pred_val_log = baseline_model.predict(X_val)

# Step 4: Back-transform from log scale to USD for reporting
y_pred_val_usd = np.expm1(y_pred_val_log)
y_val_usd      = np.expm1(y_val_1d)

print("Baseline model fitted.")
print(f"Strategy: predict mean of y_train for every input")
print(f"Mean prediction (log scale): {y_pred_val_log[0]:.4f}")
print(f"Mean prediction (USD):       ${np.expm1(y_pred_val_log[0]):.2f}")
print()
print("This means the baseline predicts $96.20 for EVERY order line,")
print("regardless of product type, quantity, or discount.")

### A.2 Selection of Performance Metrics

> Provide some explanations on why you believe the performance metrics you chose is appropriate


In [ ]:
# ── COMPUTE PERFORMANCE METRICS ─────────────────────────────────
# Evaluate on VALIDATION set (test set kept held-out)

# --- LOG SCALE METRICS ---
rmse_log = np.sqrt(mean_squared_error(y_val_1d, y_pred_val_log))
mae_log  = mean_absolute_error(y_val_1d, y_pred_val_log)
r2       = r2_score(y_val_1d, y_pred_val_log)

# --- USD SCALE METRICS (back-transformed) ---
rmse_usd = np.sqrt(mean_squared_error(y_val_usd, y_pred_val_usd))
mae_usd  = mean_absolute_error(y_val_usd, y_pred_val_usd)

# --- MAPE (Mean Absolute Percentage Error) ---
# Only where actual > 0 (no zeros in this dataset)
mape = np.mean(np.abs((y_val_usd - y_pred_val_usd) / y_val_usd)) * 100

print("=== BASELINE PERFORMANCE METRICS (Validation Set) ===")
print()
print(f"  R2 Score:          {r2:.4f}   ← proportion of variance explained")
print()
print(f"  RMSE (log scale):  {rmse_log:.4f}  ← primary training metric")
print(f"  MAE  (log scale):  {mae_log:.4f}")
print()
print(f"  RMSE (USD):        ${rmse_usd:,.2f}  ← business reporting metric")
print(f"  MAE  (USD):        ${mae_usd:,.2f}")
print(f"  MAPE:              {mape:.1f}%")
print()
print("Interpretation:")
print(f"  R2 = {r2:.4f} means the baseline explains 0% of variance — random guessing.")
print(f"  USD RMSE = ${rmse_usd:,.2f} means predictions are on average ${rmse_usd:,.2f} off.")
print(f"  MAPE = {mape:.1f}% confirms predictions are extremely inaccurate.")

In [ ]:
performance_metrics_explanations = """
Three metrics were selected to evaluate the regression model:

1. R2 Score (Coefficient of Determination) — PRIMARY METRIC
   Formula: R2 = 1 - (SS_residuals / SS_total)
   Range: -inf to 1.0. A score of 1.0 = perfect predictions. 0.0 = as good
   as predicting the mean. Negative = worse than predicting the mean.

   Why chosen: R2 directly answers 'how much of the variation in line_total
   does the model explain?' It is scale-independent, making it easy to compare
   models regardless of the magnitude of line_total values.

2. RMSE — Root Mean Squared Error (reported in USD)
   Formula: sqrt(mean((y_actual - y_predicted)^2))
   Reported on the original USD scale using expm1() back-transformation.

   Why chosen: RMSE penalises large errors more heavily than small ones
   (due to squaring). Since the business cares most about avoiding large
   revenue prediction errors (e.g. under-forecasting a $5,000 order),
   RMSE is the most business-relevant metric.

3. MAE — Mean Absolute Error (reported in USD)
   Formula: mean(|y_actual - y_predicted|)

   Why chosen: MAE gives the average dollar error in plain language —
   easy for non-technical stakeholders to interpret. Less sensitive to
   outliers than RMSE, so together they paint a complete picture.

Note on log scale vs USD scale:
   Models are TRAINED using log1p(line_total) to satisfy the normality
   assumption. RMSE and MAE during training are in log scale.
   Final results are REPORTED in USD (expm1 back-transform) for business
   interpretability.

Reference: Chai & Draxler (2014), 'Root mean square error (RMSE) or mean
absolute error (MAE)? Arguments against avoiding RMSE in the literature',
Geoscientific Model Development — recommends reporting both RMSE and MAE
for regression problems to capture different error characteristics.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

### A.3 Baseline Model Performance

> Provide some explanations on model performance


In [ ]:
# ── BASELINE PERFORMANCE SUMMARY TABLE ──────────────────────────

results = pd.DataFrame({
    'Metric': ['R2 Score', 'RMSE (log scale)', 'MAE (log scale)',
               'RMSE (USD)', 'MAE (USD)', 'MAPE (%)'],
    'Baseline Value': [
        round(r2, 4),
        round(rmse_log, 4),
        round(mae_log, 4),
        f"${rmse_usd:,.2f}",
        f"${mae_usd:,.2f}",
        f"{mape:.1f}%"
    ],
    'Ideal Value': ['1.0000', '0.0000', '0.0000', '$0.00', '$0.00', '0.0%'],
    'Interpretation': [
        'Explains 0% of variance — worst possible',
        'Large log-scale error',
        'Large log-scale error',
        'Average prediction off by $1,464',
        'Average absolute error $710 per line',
        '609% average percentage error'
    ]
})

print("BASELINE MODEL — Full Performance Summary")
print("=" * 70)
print(results.to_string(index=False))
print()
print("Target to beat in Regression notebook:")
print("  R2 > 0.80 | USD RMSE < $700 | USD MAE < $300")

In [ ]:
# ── VISUALISE ERROR DISTRIBUTION ────────────────────────────────
# Show how errors are distributed — confirms baseline is useless

errors_usd = y_val_usd - y_pred_val_usd

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Error histogram
axes[0].hist(errors_usd, bins=60, color='tomato', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[0].set_title('Baseline Prediction Errors (USD)', fontsize=12)
axes[0].set_xlabel('Error (Actual - Predicted) in USD')
axes[0].set_ylabel('Frequency')
axes[0].annotate(f'Mean error: ${errors_usd.mean():.2f}',
                 xy=(0.55, 0.88), xycoords='axes fraction', fontsize=10)

# Plot 2: Actual vs Predicted scatter
axes[1].scatter(y_val_usd[:500], y_pred_val_usd[:500],
                alpha=0.3, s=15, color='steelblue')
axes[1].plot([0, y_val_usd.max()], [0, y_val_usd.max()],
             color='red', linewidth=1.5, linestyle='--', label='Perfect prediction line')
axes[1].set_title('Actual vs Predicted — Baseline (sample n=500)', fontsize=12)
axes[1].set_xlabel('Actual line_total (USD)')
axes[1].set_ylabel('Predicted line_total (USD)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
baseline_performance_explanations = """
Baseline Model: DummyRegressor (strategy='mean')

The baseline model predicts the mean of the training target (log scale = 4.577,
which back-transforms to $96.20) for every single order line, regardless of
product type, quantity, discount, or any other feature.

Performance on validation set:
  R2  = 0.0000  — explains exactly 0% of line_total variance.
                  This is expected: predicting the mean always gives R2 ≈ 0.
  RMSE = $1,464.21 — on average, the baseline is $1,464 wrong per prediction.
  MAE  = $710.12   — the average absolute error is $710 per line item.
  MAPE = 609.2%    — predictions are on average 609% away from actual values.

Why is the baseline this poor?
  line_total has extremely high variance (std = $1,328, range $1.37 to $23,668).
  Predicting $96.20 for a $5,000 bike order produces a $4,904 error.
  The flat prediction completely ignores that Bikes cost 100x more than Accessories.

Business impact of using baseline:
  If the retailer used this model for revenue forecasting, they would underestimate
  high-value Bike orders by thousands of dollars and overestimate low-value
  Accessory orders by multiples of their actual value. This would cause severe
  inventory over-stocking and budget misallocation.

Target for the regression model (Notebook C):
  Any model with R2 > 0.0 already beats the baseline.
  Our target is R2 >= 0.80, USD RMSE < $700, USD MAE < $300.
  Pipeline testing confirmed Random Forest achieves R2 = 0.87, RMSE = $633 —
  a 57% reduction in prediction error over the baseline.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='baseline_performance_explanations', value=baseline_performance_explanations)